# Avito Retrieval Contest

```text
данные
→ два BM25-поиска
→ два E5-поиска
→ объединение кандидатов через RRF
→ бонус за локацию и точный заголовок
→ top-50
→ answer.csv
```

## 1. Настройки и данные

`CANDIDATE_DEPTH = 300` — сколько объявлений возвращает каждый retriever.  
`TOP_K = 50` — сколько объявлений остаётся в ответе.

In [20]:
import re
from collections import defaultdict
from pathlib import Path

import bm25s
import numpy as np
import polars as pl
import torch

DATASET_DIR = Path("dataset")
ARTIFACTS_DIR = Path("artifacts")
ANSWER_PATH = Path("answer.csv")

CANDIDATE_DEPTH = 300
TOP_K = 50
E5_MODEL = "intfloat/multilingual-e5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

queries = pl.read_parquet(DATASET_DIR / "benchmark_queries.parquet")
items = pl.read_parquet(DATASET_DIR / "benchmark_items.parquet")


def clean(value, lower=True):
    text = re.sub(r"\s+", " ", value or "").strip()
    return text.lower().replace("ё", "е") if lower else text


print(f"queries: {queries.height}, items: {items.height}, device: {DEVICE}")

queries: 2452, items: 189212, device: cpu


## 2. BM25

Используются две отдельные выдачи:

- `title_params`: заголовок и параметры объявления против запроса и его параметров;
- `description`: описание объявления против короткого запроса.

Если готовый файл кандидатов существует, он просто загружается. Иначе загружаются или строятся два BM25-индекса.

In [13]:
sparse_path = ARTIFACTS_DIR / f"benchmark_sparse_candidates_k{CANDIDATE_DEPTH}.npz"

if sparse_path.exists():
    with np.load(sparse_path) as saved:
        sparse = {
            "title_params": saved["title_params"],
            "description": saved["description"],
        }
    print("BM25 candidates loaded")
else:
    titles = [clean(x) for x in items["item_title_raw"]]
    item_params = [clean(x) for x in items["item_infm_params_text"]]
    query_texts = [clean(x) for x in queries["search_query"]]
    query_params = [clean(x) for x in queries["search_infm_params_text"]]

    bm25_data = {
        "title_params": (
            [f"{title} {params}".strip() for title, params in zip(titles, item_params)],
            [f"{query} {params}".strip() for query, params in zip(query_texts, query_params)],
        ),
        "description": (
            [clean(x) for x in items["item_description_raw"]],
            query_texts,
        ),
    }

    sparse = {}
    for name, (corpus, channel_queries) in bm25_data.items():
        index_dir = ARTIFACTS_DIR / "bm25" / name

        if index_dir.exists():
            model = bm25s.BM25.load(
                index_dir, load_corpus=False, mmap=True, show_progress=False
            )
        else:
            model = bm25s.BM25()
            model.index(bm25s.tokenize(corpus, stopwords=None, stemmer=None))
            model.save(index_dir, show_progress=False)

        indices, _ = model.retrieve(
            bm25s.tokenize(channel_queries, stopwords=None, stemmer=None),
            k=CANDIDATE_DEPTH,
        )
        sparse[name] = indices.astype(np.int32)

    np.savez_compressed(sparse_path, **sparse)
    print("BM25 candidates built and saved")

BM25 candidates loaded


## 3. E5 и FAISS

Объявление кодируется как:

```text
passage: title. params. description
```

Для каждого запроса строятся два представления:

```text
query: search_query
query: search_query. search_params
```

Готовые dense-кандидаты загружаются сразу. Если их нет, notebook использует сохранённые embeddings и FAISS-индекс либо строит недостающие файлы.

In [14]:
import faiss
from sentence_transformers import SentenceTransformer

embeddings_path = ARTIFACTS_DIR / "e5_benchmark_item_embeddings.npy"
ids_path = ARTIFACTS_DIR / "e5_benchmark_item_ids.parquet"
index_path = ARTIFACTS_DIR / "e5_benchmark_faiss.index"
dense_path = ARTIFACTS_DIR / f"e5_benchmark_candidates_k{CANDIDATE_DEPTH}.npz"

same_item_order = (
    ids_path.exists()
    and pl.read_parquet(ids_path)["item_id"].to_list() == items["item_id"].to_list()
)

if dense_path.exists() and same_item_order:
    with np.load(dense_path) as saved:
        dense = {
            "dense_short": saved["dense_short"],
            "dense_full": saved["dense_full"],
        }
    print("E5 candidates loaded")
else:
    model = SentenceTransformer(E5_MODEL, device=DEVICE)
    model.max_seq_length = 256

    if embeddings_path.exists() and same_item_order:
        item_embeddings = np.load(embeddings_path, mmap_mode="r")
    else:
        passages = [
            f"passage: {clean(title, False)}. {clean(params, False)}. {clean(description, False)}"
            for title, params, description in items.select(
                "item_title_raw",
                "item_infm_params_text",
                "item_description_raw",
            ).iter_rows()
        ]
        item_embeddings = model.encode(
            passages,
            batch_size=32,
            show_progress_bar=True,
            normalize_embeddings=True,
        ).astype(np.float32)
        np.save(embeddings_path, item_embeddings)
        items.select("item_id").write_parquet(ids_path)

    if index_path.exists() and same_item_order:
        index = faiss.read_index(str(index_path))
    else:
        index = faiss.IndexFlatIP(item_embeddings.shape[1])
        index.add(np.ascontiguousarray(item_embeddings))
        faiss.write_index(index, str(index_path))

    short_queries = [
        f"query: {clean(query, False)}"
        for query in queries["search_query"]
    ]
    full_queries = [
        f"query: {clean(query, False)}. {clean(params, False)}"
        for query, params in queries.select(
            "search_query", "search_infm_params_text"
        ).iter_rows()
    ]
    query_embeddings = model.encode(
        short_queries + full_queries,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    _, indices = index.search(query_embeddings, CANDIDATE_DEPTH)
    split = queries.height
    dense = {
        "dense_short": indices[:split].astype(np.int32),
        "dense_full": indices[split:].astype(np.int32),
    }
    np.savez_compressed(dense_path, **dense)
    print("E5 candidates built and saved")

E5 candidates loaded


Подберём параметры через optuna:

In [17]:
import optuna

In [21]:
train = pl.read_parquet(DATASET_DIR / "train.parquet")

QUERY_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

tuning_queries = (
    train
    .join(
        items.select("item_id"),
        on="item_id",
        how="semi",
    )
    .group_by(
        QUERY_COLUMNS,
        maintain_order=True,
    )
    .agg(
        pl.col("item_id")
        .unique()
        .alias("positive_items")
    )
    .sample(
        n=2000,
        seed=42,
        shuffle=True,
    )
)

print(f"tuning queries: {tuning_queries.height}")

tuning queries: 2000


In [23]:
TUNING_COUNT = tuning_queries.height

tuning_candidates_path = ARTIFACTS_DIR / f"tuning_candidates_q{TUNING_COUNT}_k{CANDIDATE_DEPTH}.npz"

if tuning_candidates_path.exists():
    with np.load(tuning_candidates_path) as saved:
        tuning_candidates = {
            "title_params": saved["title_params"],
            "description": saved["description"],
            "dense_short": saved["dense_short"],
            "dense_full": saved["dense_full"],
        }
    print("Tuning candidates loaded")

else:
    tuning_texts = [clean(value) for value in tuning_queries["search_query"]]
    tuning_params = [clean(value) for value in tuning_queries["search_infm_params_text"]]
    title_params_queries = [f"{query} {params}".strip() for query, params in zip(tuning_texts, tuning_params)]

    title_params_model = bm25s.BM25.load(
        ARTIFACTS_DIR / "bm25" / "title_params",
        load_corpus=False,
        mmap=True,
        show_progress=False,
    )

    title_params_indices, _ = title_params_model.retrieve(
        bm25s.tokenize(title_params_queries,stopwords=None,stemmer=None),
        k=CANDIDATE_DEPTH,
    )

    description_model = bm25s.BM25.load(
        ARTIFACTS_DIR / "bm25" / "description",
        load_corpus=False,
        mmap=True,
        show_progress=False,
    )

    description_indices, _ = description_model.retrieve(
        bm25s.tokenize(tuning_texts,stopwords=None,stemmer=None),
        k=CANDIDATE_DEPTH,
    )

    tuning_model = SentenceTransformer(E5_MODEL, device=DEVICE)
    tuning_model.max_seq_length = 256
    
    short_queries = [f"query: {clean(query, False)}"for query in tuning_queries["search_query"]]
    full_queries = [
        f"query: {clean(query, False)}. {clean(params, False)}"
        for query, params in tuning_queries.select(
            "search_query",
            "search_infm_params_text",
        ).iter_rows()]

    tuning_embeddings = tuning_model.encode(
        short_queries + full_queries,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    tuning_index = faiss.read_index(str(index_path))

    _, dense_indices = tuning_index.search(tuning_embeddings,CANDIDATE_DEPTH)

    tuning_candidates = {
        "title_params": title_params_indices.astype(np.int32),
        "description": description_indices.astype(np.int32),
        "dense_short": dense_indices[:TUNING_COUNT].astype(np.int32),
        "dense_full": dense_indices[TUNING_COUNT:].astype(np.int32),
    }
    np.savez_compressed(tuning_candidates_path, **tuning_candidates)
    print("Tuning candidates built and saved")

Split strings:   0%|          | 0/2000 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/2000 [00:00<?, ?it/s]

Split strings:   0%|          | 0/2000 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/2000 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Tuning candidates built and saved


In [24]:
CHANNELS = [
    "title_params",
    "description",
    "dense_short",
    "dense_full",
]

item_ids = items["item_id"].to_list()
item_locations = items["item_location_id"].to_list()
item_titles = [clean(value) for value in items["item_title_raw"]]
tuning_data = []

for query_index in range(tuning_queries.height):
    rows = {}

    for channel_index, channel in enumerate(CHANNELS):
        for rank, item_index in enumerate(tuning_candidates[channel][query_index], start=1):
            item_index = int(item_index)
            if item_index not in rows:
                rows[item_index] = [.0] * 4

            rows[item_index][channel_index] = 1.0 / (18 + rank)

    candidate_indices = np.array(list(rows), dtype=np.int32)
    reciprocal_ranks = np.array(list(rows.values()), dtype=np.float32)

    query_location = tuning_queries["search_location_id"][query_index]
    query_text = clean(tuning_queries["search_query"][query_index])

    metadata_bonus = np.array([0.4 * (item_locations[item_index] == query_location)
            + 0.2 * (item_titles[item_index] == query_text)
            for item_index in candidate_indices
        ],dtype=np.float32)

    positive_items = set(tuning_queries["positive_items"][query_index])
    relevant = np.array([item_ids[item_index] in positive_items for item_index in candidate_indices], dtype=bool,)

    tuning_data.append((reciprocal_ranks, metadata_bonus, relevant, len(positive_items)))

print(f"prepared queries: {len(tuning_data)}")

prepared queries: 2000


In [26]:
def fusion_recall(weights):
    """Recall@50"""
    weights = np.array(
        [weights["title_params"], weights["description"], weights["dense_short"], weights["dense_full"]],
        dtype=np.float32
    )

    recalls = []
    for (reciprocal_ranks, metadata_bonus, relevant, positive_count) in tuning_data:
        scores = reciprocal_ranks @ weights + metadata_bonus

        top_50 = np.argsort(-scores, kind="stable")[:TOP_K]

        recall = relevant[top_50].sum() / positive_count
        recalls.append(recall)
    return float(np.mean(recalls))

In [28]:
# подбирались ранее
WEIGHTS = {
    "title_params": 2.5425732070511984,
    "description": 3.910998189448521,
    "dense_short": 2.2,
    "dense_full": 4.415302552589752,
}

old_recall = fusion_recall(OLD_WEIGHTS)

print(f"Old weights Recall@50: {old_recall:.5f}")

Old weights Recall@50: 0.66572


In [33]:
def objective(trial):
    weights = {
        "title_params": trial.suggest_float("title_params", 0.5, 6.0),
        "description": trial.suggest_float("description", 0.5, 6.0),
        "dense_short": trial.suggest_float("dense_short", 0.5, 6.0),
        "dense_full": trial.suggest_float("dense_full", 0.5, 6.0),
    }
    return fusion_recall(weights)

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.HyperbandPruner(),
)

study.optimize(objective, n_trials=10000, n_jobs=1, show_progress_bar=True)

print(f"Old Recall@50:  {old_recall:.5f}")
print(f"Best Recall@50: {study.best_value:.5f}")
print(study.best_params)

[I 2026-09-20 18:35:06,741] A new study created in memory with name: no-name-da8f779a-4ee5-4ed9-85a8-be854e29d94d


  0%|          | 0/10000 [00:00<?, ?it/s]

[I 2026-09-20 18:35:06,842] Trial 0 finished with value: 0.6621482142857142 and parameters: {'title_params': 2.5599706536604936, 'description': 5.728928685254539, 'dense_short': 4.525966679962728, 'dense_full': 3.7926216630837013}. Best is trial 0 with value: 0.6621482142857142.
[I 2026-09-20 18:35:06,910] Trial 1 finished with value: 0.6596571428571428 and parameters: {'title_params': 1.358102522433401, 'description': 1.3579698618491145, 'dense_short': 0.819459866925097, 'dense_full': 5.263968801762143}. Best is trial 0 with value: 0.6621482142857142.
[I 2026-09-20 18:35:06,980] Trial 2 finished with value: 0.665382142857143 and parameters: {'title_params': 3.8061325645876485, 'description': 4.394399177878251, 'dense_short': 0.6132147186269135, 'dense_full': 5.834504186890968}. Best is trial 2 with value: 0.665382142857143.
[I 2026-09-20 18:35:07,050] Trial 3 finished with value: 0.6597744047619049 and parameters: {'title_params': 5.0784345244023195, 'description': 1.6678651087305187,

In [ ]:
if old_recall < study.best_value:
    WEIGHTS = study.best_params
    print("weights updated")

## 4. Объединение выдач

Для каждого объявления складываются вклады четырёх каналов:

```text
weight / (18 + rank)
```

После RRF добавляется `0.4` при совпадении локации и `0.2` при точном совпадении нормализованного запроса с заголовком. Бонусы применяются ко всему объединению, и только потом выбирается top-50.

In [15]:
candidates = {**sparse, **dense}

item_ids = items["item_id"].to_list()
item_locations = items["item_location_id"].to_list()
query_locations = queries["search_location_id"].to_list()
item_titles = [clean(x) for x in items["item_title_raw"]]
query_texts = [clean(x) for x in queries["search_query"]]

predictions = []

for query_index in range(queries.height):
    scores = defaultdict(float)

    for channel, weight in WEIGHTS.items():
        for rank, item_index in enumerate(candidates[channel][query_index], start=1):
            scores[int(item_index)] += weight / (18 + rank)

    for item_index in scores:
        if item_locations[item_index] == query_locations[query_index]:
            scores[item_index] += 0.4
        if item_titles[item_index] == query_texts[query_index]:
            scores[item_index] += 0.2

    top_items = sorted(scores, key=scores.get, reverse=True)[:TOP_K]
    predictions.append([item_ids[item_index] for item_index in top_items])

## 5. `answer.csv`

В каждой строке сохраняются `query_id` и 50 идентификаторов объявлений, разделённых пробелами.

In [16]:
answer = pl.DataFrame(
    {
        "query_id": queries["query_id"],
        "answer": [" ".join(row) for row in predictions],
    }
)
answer.write_csv(ANSWER_PATH)

print(f"saved: {ANSWER_PATH}")
print(f"rows: {answer.height}, items per query: {len(predictions[0])}")
answer.head()

saved: answer.csv
rows: 2452, items per query: 50


query_id,answer
str,str
"""70DfDUpwjxB4lzFd""","""255fbeaf526a1cc1 355392014208b…"
"""JTrdTaZJvSiLPkXj""","""422d3ffdd5bbf626 cbeccbecb1fb8…"
"""LZCZNoVG4AFUkVRJ""","""dab52187b4500d9b 168a9207e80b0…"
"""660ac9QVtXkRxZC3""","""a846a2a5241e1180 af91ec4a29b66…"
"""YgHcM9MVbxKnxD1e""","""ba78ad593ec3412d 23bc4a9372c16…"
